Universidad de Antioquia — Algoritmia y Programación

La Reserva

Wbuanderley Ramírez Martínez · Disney Restrepo Enciso

In [1]:
import sys
!{sys.executable} -m pip install reportlab fpdf

In [2]:
import csv
import datetime
import os
import re
import time
from fpdf import FPDF

try:
    from IPython.display import clear_output, display, FileLink
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# Rutas base — compatibles con Google Colab y ejecucion local
BASE_DIR      = '/content' if os.path.exists('/content') else os.path.dirname(os.path.abspath('main.py'))
DATA_DIR      = os.path.join(BASE_DIR, 'data')
DOC_DIR       = os.path.join(BASE_DIR, 'doc')
USUARIOS_CSV  = os.path.join(DATA_DIR, 'usuarios.csv')
ITEMS_CSV     = os.path.join(DATA_DIR, 'items.csv')
PRESTAMOS_CSV = os.path.join(DATA_DIR, 'prestamos.csv')
ADMINS_CSV    = os.path.join(DATA_DIR, 'admins.csv')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(DOC_DIR,  exist_ok=True)


In [3]:
# ─────────────────────────────────────────────────────────────────
#  CLASES
# ─────────────────────────────────────────────────────────────────

# Install dejavu fonts for FPDF in Colab if not already present
import sys
EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    # Using -qq for quiet mode, and checking for existance is usually not needed as apt-get handles it
    # This ensures the font is available for FPDF's Unicode support.
    !apt-get update -qq && apt-get install -qq fonts-dejavu

class clsUsuarios:
    """Representa un usuario/amigo registrado en La Reserva."""

    def __init__(self, nombre, apellido, documento, correo, tiempo):
        self.nombre    = nombre
        self.apellido  = apellido
        self.documento = documento
        self.correo    = correo
        self.tiempo    = int(tiempo)

    def to_dict(self):
        return {"nombre": self.nombre, "apellido": self.apellido,
                "documento": self.documento, "correo": self.correo,
                "tiempo": self.tiempo}

    @classmethod
    def from_dict(cls, d):
        return cls(d["nombre"], d["apellido"], d["documento"],
                   d["correo"], d["tiempo"])

    def __str__(self):
        return (f"{self.nombre} {self.apellido} | Doc: {self.documento} "
                f"| Correo: {self.correo} | Dias: {self.tiempo}")


class clsPrestamo:
    """Representa un prestamo activo o historico."""

    def __init__(self, documento, item_id, fecha,
                 devuelto="No", fecha_devolucion="", vendido="No"):
        self.documento        = documento
        self.item_id          = item_id
        self.fecha            = fecha
        self.devuelto         = devuelto
        self.fecha_devolucion = fecha_devolucion
        self.vendido          = vendido

    def dias_prestado(self):
        inicio = datetime.datetime.strptime(self.fecha, "%Y-%m-%d").date()
        return (datetime.date.today() - inicio).days

    def to_dict(self):
        return {"documento": self.documento, "item_id": self.item_id,
                "fecha": self.fecha, "devuelto": self.devuelto,
                "fecha_devolucion": self.fecha_devolucion,
                "vendido": self.vendido}

    @classmethod
    def from_dict(cls, d):
        return cls(d["documento"], d["item_id"], d["fecha"],
                   d.get("devuelto", "No"), d.get("fecha_devolucion", ""),
                   d.get("vendido", "No"))


class clsItem:
    """Representa un item disponible para prestamo."""

    def __init__(self, nombre, categoria, precio, estado, item_id, disponible="Si"):
        self.nombre     = nombre
        self.categoria  = categoria
        self.precio     = float(precio)
        self.estado     = estado
        self.item_id    = item_id
        self.disponible = disponible

    def estado_difuso(self):
        """Logica difusa: convierte etiqueta de estado a valor 0.0-1.0."""
        tabla = {"Excelente": 1.0, "Bueno": 0.75,
                 "Regular": 0.5,   "Malo": 0.25, "Pesimo": 0.1}
        return tabla.get(self.estado, 0.5)

    def to_dict(self):
        return {"nombre": self.nombre, "categoria": self.categoria,
                "precio": self.precio, "estado": self.estado,
                "item_id": self.item_id, "disponible": self.disponible}

    @classmethod
    def from_dict(cls, d):
        return cls(d["nombre"], d["categoria"], d["precio"],
                   d["estado"], d["item_id"], d.get("disponible", "Si"))

    def __str__(self):
        disp = "Disponible" if self.disponible == "Si" else "Prestado  "
        return (f"[{self.item_id}] {self.nombre:<22} | {self.categoria:<18} "
                f"| {self.estado:<10} ({self.estado_difuso():.0%}) "
                f"| ${self.precio:>10,.0f} | {disp}")


# ─────────────────────────────────────────────────────────────────
#  PERSISTENCIA CSV
# ─────────────────────────────────────────────────────────────────

CAMPOS_USU   = ["nombre", "apellido", "documento", "correo", "tiempo"]
CAMPOS_ITEM  = ["nombre", "categoria", "precio", "estado", "item_id", "disponible"]
CAMPOS_PREST = ["documento", "item_id", "fecha", "devuelto",
                "fecha_devolucion", "vendido"]
CAMPOS_ADM   = ["usuario", "clave"]


def _leer_csv(path):
    if not os.path.exists(path):
        return []
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def _escribir_csv(path, campos, datos):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=campos)
        w.writeheader()
        w.writerows(datos)

def cargar_usuarios():
    return [clsUsuarios.from_dict(r) for r in _leer_csv(USUARIOS_CSV)]

def guardar_usuarios(lista):
    _escribir_csv(USUARIOS_CSV, CAMPOS_USU, [u.to_dict() for u in lista])

def cargar_items():
    return [clsItem.from_dict(r) for r in _leer_csv(ITEMS_CSV)]

def guardar_items(lista):
    _escribir_csv(ITEMS_CSV, CAMPOS_ITEM, [i.to_dict() for i in lista])

def cargar_prestamos():
    return [clsPrestamo.from_dict(r) for r in _leer_csv(PRESTAMOS_CSV)]

def guardar_prestamos(lista):
    _escribir_csv(PRESTAMOS_CSV, CAMPOS_PREST, [p.to_dict() for p in lista])

def cargar_admins():
    """
    Administradores validos de La Reserva.
    Usuarios: Disney, Wbuanderley, Kevin, John
    Contrasena para todos: lareserva2026
    Se regenera el CSV siempre para garantizar datos correctos.
    """
    ADMINS_VALIDOS = [
        {"usuario": "Disney",      "clave": "lareserva2026"},
        {"usuario": "Wbuanderley", "clave": "lareserva2026"},
        {"usuario": "John",        "clave": "lareserva2026"},
    ]
    # Siempre reescribir para garantizar que esten los 4 usuarios
    with open(ADMINS_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["usuario", "clave"])
        w.writeheader()
        w.writerows(ADMINS_VALIDOS)
    # Retornar como diccionario usuario -> clave
    return {a["usuario"]: a["clave"] for a in ADMINS_VALIDOS}


# ─────────────────────────────────────────────────────────────────
#  SEED: items iniciales de La Reserva
# ─────────────────────────────────────────────────────────────────

ITEMS_INICIALES = [
    clsItem("FIFA23",           "Videojuegos",    250000, "Excelente", "VID001"),
    clsItem("CallOfDuty",       "Videojuegos",    300000, "Bueno",     "VID002"),
    clsItem("MarioKart",        "Videojuegos",    200000, "Regular",   "VID003"),
    clsItem("PythonBasico",     "Libros",          50000, "Excelente", "LIB001"),
    clsItem("AlgebraLineal",    "Libros",          60000, "Bueno",     "LIB002"),
    clsItem("CienAnos",         "Libros",          45000, "Regular",   "LIB003"),
    clsItem("ThrillerMJ",       "Musica y video",  30000, "Excelente", "MUS001"),
    clsItem("ConciertoRock",    "Musica y video",  35000, "Bueno",     "MUS002"),
    clsItem("PeliculaAvengers", "Musica y video",  40000, "Excelente", "MUS003"),
    clsItem("TaladroBosch",     "Herramientas",   150000, "Bueno",     "HER001"),
    clsItem("MartilloPro",      "Herramientas",    30000, "Excelente", "HER002"),
    clsItem("LlaveInglesa",     "Herramientas",    25000, "Regular",   "HER003"),
    clsItem("Prestamo100k",     "Dinero",         100000, "Bueno",     "DIN001"),
    clsItem("Prestamo200k",     "Dinero",         200000, "Bueno",     "DIN002"),
    clsItem("Cafetera",         "Miscelaneo",     120000, "Excelente", "MIS001"),
    clsItem("Ventilador",       "Miscelaneo",      90000, "Bueno",     "MIS002"),
    clsItem("LaptopHP",         "Miscelaneo",    1500000, "Excelente", "MIS003"),
    clsItem("MouseGamer",       "Miscelaneo",      80000, "Bueno",     "MIS004"),
    clsItem("TecladoRGB",       "Miscelaneo",     120000, "Excelente", "MIS005"),
    clsItem("AudifonosSony",    "Miscelaneo",     200000, "Bueno",     "MIS006"),
]

def inicializar_items():
    if not os.path.exists(ITEMS_CSV) or os.path.getsize(ITEMS_CSV) == 0:
        guardar_items(ITEMS_INICIALES)


# ─────────────────────────────────────────────────────────────────
#  VALIDACIONES
# ─────────────────────────────────────────────────────────────────

def validar_nombre(nombre):
    if len(nombre) < 3:
        return False, "Debe tener al menos 3 letras."
    if any(c.isdigit() for c in nombre):
        return False, "No puede contener numeros."
    return True, ""

def validar_documento(doc):
    if not doc.isdigit():
        return False, "Solo se permiten numeros."
    if not (3 <= len(doc) <= 15):
        return False, "Debe tener entre 3 y 15 digitos."
    return True, ""

def validar_correo(correo):
    if not re.match(r'^[^@\s]+@[^@\s]+\.[cC][oO][mM]$', correo):
        return False, "Debe contener @ y terminar en .com"
    return True, ""

def validar_tiempo(t):
    if t not in ("5", "10", "15", "30"):
        return False, "Solo se permiten: 5, 10, 15 o 30 dias."
    return True, ""

def pedir_campo(mensaje, validador):
    while True:
        valor = input(mensaje).strip()
        ok, err = validador(valor)
        if ok:
            return valor
        print(f"  >> Error: {err}")


# ─────────────────────────────────────────────────────────────────
#  PANTALLA
# ─────────────────────────────────────────────────────────────────

LINEA = '=' * 64
SEPARADOR = '-' * 64

BANNER_TXT = r"""
  ██╗      █████╗     ██████╗ ███████╗███████╗███████╗██████╗ ██╗   ██╗ █████╗
  ██║     ██╔══██╗    ██╔══██╗██╔════╝██╔════╝██╔════╝██╔══██╗██║   ██║██╔══██╗
  ██║     ███████║    ██████╔╝█████╗  ███████╗█████╗  ██████╔╝██║   ██║███████║
  ██║     ██╔══██║    ██╔══██╗██╔══╝  ╚════██║██╔══╝  ██╔══██╗╚██╗ ██╔╝██╔══██║
  ███████╗██║  ██║    ██║  ██║███████╗███████║███████╗██║  ██║  ╚████╔╝██║  ██║
  ╚══════╝╚═╝  ╚═╝    ╚═╝  ╚═╝╚══════╝╚══════╝╚══════╝╚═╝  ╚═╝   ╚═══╝ ╚═╝  ╚═╝
"""

def limpiar():
    os.system("cls" if os.name == "nt" else "clear")

def banner():
    """Muestra el de La Reserva"""
    limpiar()
    print()
    for linea in BANNER_TXT.split('\n'): # Corrected: Changed BANNER to BANNER_TXT
        print(linea)
    print()
    print(LINEA)
    print("  Bienvenido a La Reserva")
    print(LINEA)

def titulo(texto):
    print(f"\n{LINEA}")
    print(f"  {texto}")
    print(LINEA)

def pausa():
    input("\n  Presiona ENTER para continuar...")

from fpdf import FPDF # Added import statement
from IPython.display import FileLink, display # Added imports for Colab specific functions

# Helper functions for PDF generation
def _get_font_path():
    """Determina la ruta de la fuente para FPDF en Colab o local."""
    if EN_COLAB:
        # Colab a menudo tiene fuentes DejaVu disponibles
        # After installing fonts-dejavu, this path should be valid.
        return '/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf'
    else:
        # Para ejecucion local, FPDF puede encontrar fuentes estandar o se puede especificar una ruta local
        return None # Usara las fuentes por defecto de FPDF (e.g., Helvetica)


def _mostrar_link_pdf(ruta_pdf):
    if EN_COLAB:
        display(FileLink(ruta_pdf))
    else:
        print(f"  PDF generado en: {ruta_pdf}")


class PDFBase(FPDF):
    def __init__(self):
        super().__init__()
        # Make sure the font path is valid. If _get_font_path returns None, it will use FPDF's default fonts.
        font_path = _get_font_path()
        if font_path and os.path.exists(font_path):
            self.add_font('DejaVu', '', font_path, uni=True) # Custom font for Unicode support
            self.set_font('DejaVu', size=12)
        else:
            # Fallback to a standard FPDF font if DejaVu is not available
            self.set_font('Helvetica', size=12)
        self.set_text_color(20, 20, 20)

    def header(self):
        # Use 'Helvetica' as a safe fallback if 'DejaVu' isn't loaded
        if 'DejaVu' in self.fonts:
            self.set_font('DejaVu', size=10)
        else:
            self.set_font('Helvetica', size=10)
        self.cell(0, 10, 'La Reserva - Certificado Oficial', 0, 1, 'C')
        self.ln(5)

    def footer(self):
        if 'DejaVu' in self.fonts:
            self.set_font('DejaVu', size=8)
        else:
            self.set_font('Helvetica', size=8)
        self.set_y(-15)
        self.set_text_color(128)
        self.cell(0, 10, f'Pagina {self.page_no()}/{{nb}}', 0, 0, 'C')

    def titulo_seccion(self, titulo):
        if 'DejaVu' in self.fonts:
            self.set_font('DejaVu', size=14)
        else:
            self.set_font('Helvetica', size=14)
        self.set_text_color(0, 0, 0)
        self.ln(5)
        self.cell(0, 10, titulo, 0, 1, 'L')
        self.set_line_width(0.5)
        # Adjust the line length based on text width for accuracy
        self.line(self.get_x(), self.get_y(), self.get_x() + self.get_string_width(titulo) + 5, self.get_y())
        self.ln(5)

    def fila_datos(self, etiqueta, valor):
        if 'DejaVu' in self.fonts:
            self.set_font('DejaVu', size=11)
        else:
            self.set_font('Helvetica', size=11)
        self.set_text_color(50, 50, 50)
        self.cell(50, 7, etiqueta, 0, 0, 'L')
        if 'DejaVu' in self.fonts:
            self.set_font('DejaVu', size=11)
        else:
            self.set_font('Helvetica', size=11)
        self.set_text_color(0, 0, 0)
        self.multi_cell(0, 7, str(valor), 0, 'L')


# ─────────────────────────────────────────────────────────────────
#  MODULO 1: REGISTRAR USUARIO
# ─────────────────────────────────────────────────────────────────

def registrar_usuario():
    titulo("REGISTRAR USUARIO")
    usuarios = cargar_usuarios()

    nombre   = pedir_campo("  Nombre    : ", validar_nombre)
    apellido = pedir_campo("  Apellido  : ", validar_nombre)
    doc      = pedir_campo("  Documento : ", validar_documento)

    if any(u.documento == doc for u in usuarios):
        print("  >> Ya existe un usuario con ese documento.")
        pausa(); return

    correo = pedir_campo("  Correo    : ", validar_correo)
    tiempo = pedir_campo("  Dias de prestamo (5 / 10 / 15 / 30): ", validar_tiempo)

    usuarios.append(clsUsuarios(nombre, apellido, doc, correo, tiempo))
    guardar_usuarios(usuarios)
    print(f"\n  OK - Usuario '{nombre} {apellido}' registrado correctamente.")
    pausa()


# ─────────────────────────────────────────────────────────────────
#  MODULO 2: REGISTRAR PRESTAMO
# ─────────────────────────────────────────────────────────────────

def registrar_prestamo():
    titulo("REGISTRAR PRESTAMO")
    usuarios = cargar_usuarios()
    items = cargar_items()
    prestamos = cargar_prestamos()

    doc = input(" Documento del usuario: ").strip()
    usuario = next((u for u in usuarios if u.documento == doc), None)
    if not usuario:
        print(" >> Usuario NO encontrado.")
        print(" >> Por favor registrelo primero (opcion 1).")
        pausa(); return

    print(f"\n Usuario: {usuario.nombre} {usuario.apellido}"
          f" | Dias permitidos: {usuario.tiempo}")

    disponibles = [i for i in items if i.disponible == "Si"]
    if not disponibles:
        print(" >> No hay items disponibles en este momento.")
        pausa(); return

    print(f"\n {'ID':<10} {'Nombre':<22} {'Categoria':<18}"
          f" {'Estado':<10} {'Calidad':>8} {'Precio':>12}")
    print(" " + "-" * 84)
    for it in disponibles:
        print(f" {it.item_id:<10} {it.nombre:<22} {it.categoria:<18}"
              f" {it.estado:<10} {it.estado_difuso():>7.0%}"
              f" ${it.precio:>11,.0f}")

    item_id = input("\n Ingresa el ID del item a prestar: ").strip().upper()
    item = next((i for i in disponibles if i.item_id == item_id), None)
    if not item:
        print(" >> ID de item invalido o no disponible.")
        pausa(); return

    # Registrar Préstamo
    fecha_hoy = datetime.date.today().strftime("%Y-%m-%d")
    nuevo_prestamo = clsPrestamo(documento=usuario.documento, item_id=item.item_id, fecha=fecha_hoy)
    prestamos.append(nuevo_prestamo)

    # Actualizar disponibilidad del item
    for i in items:
        if i.item_id == item.item_id:
            i.disponible = "No"

    guardar_prestamos(prestamos)
    guardar_items(items)

    print("\n OK - Prestamo registrado exitosamente:")
    print(f"      Item    : {item.nombre} [{item.item_id}]")
    print(f"      Usuario : {usuario.nombre} {usuario.apellido}")
    print(f"      Fecha   : {fecha_hoy}  |  Vence en {usuario.tiempo} dias")

    # ================================================================
    # GENERACIÓN DEL CERTIFICADO PDF (Usa fuentes compatibles internamente)
    # ================================================================
    print("\n Generando certificado de prestamo...")
    nombre_arch = f"Certificado_Prestamo_{usuario.nombre}_{usuario.apellido}_{item.item_id}_{fecha_hoy}.pdf"
    ruta_pdf = os.path.join(DOC_DIR, nombre_arch)

    f_fuente = 'DejaVu' if _get_font_path() else 'Helvetica'

    pdf = PDFBase()
    pdf.add_page()
    pdf.titulo_seccion('CERTIFICADO DE PRESTAMO VALIDO')
    pdf.ln(2)
    pdf.fila_datos('Fecha Prestamo:', fecha_hoy)
    pdf.fila_datos('Plazo Autorizado:', f"{usuario.tiempo} dias")
    pdf.ln(3)
    pdf.titulo_seccion('DATOS DEL USUARIO BENEFICIARIO')
    pdf.fila_datos('Nombre Completo:', f"{usuario.nombre} {usuario.apellido}")
    pdf.fila_datos('Documento de Identidad:', usuario.documento)
    pdf.fila_datos('Correo Electronico:', usuario.correo)
    pdf.ln(3)
    pdf.titulo_seccion('DETALLES DEL OBJETO / ITEM PRESTADO')
    pdf.fila_datos('Nombre del Item:', item.nombre)
    pdf.fila_datos('Identificador ID:', item.item_id)
    pdf.fila_datos('Categoria:', item.categoria)
    pdf.fila_datos('Estado de Entrega:', f"{item.estado} ({item.estado_difuso():.0%})")
    pdf.fila_datos('Valor de Referencia:', f"${item.precio:,.0f}")
    pdf.ln(5)
    pdf.set_font(f_fuente, size=10)
    pdf.set_text_color(30, 30, 30)
    pdf.multi_cell(0, 6, 'Este documento oficial certifica que el usuario ha solicitado el articulo en la fecha indicada de manera conforme bajo los reglamentos de La Reserva.')

    try:
        pdf.output(ruta_pdf)
        _mostrar_link_pdf(ruta_pdf)
    except Exception as e:
        print(f" >> Error al guardar el PDF: {e}")

    # Pausa externa obligatoria para retornar controladamente al menú
    print("")
    pausa()

# ─────────────────────────────────────────────────────────────────
#  MODULO 3: REGISTRAR DEVOLUCION
# ─────────────────────────────────────────────────────────────────

def registrar_devolucion():
    titulo("REGISTRAR DEVOLUCION")
    usuarios = cargar_usuarios()
    items = cargar_items()
    prestamos = cargar_prestamos()

    # Validar que el usuario exista
    doc = input(" Documento del usuario: ").strip()
    usuario = next((u for u in usuarios if u.documento == doc), None)
    if not usuario:
        print(" >> Usuario NO encontrado.")
        pausa(); return

    # Buscar préstamos activos del usuario
    activos = [p for p in prestamos if p.documento == doc and p.devuelto == "No" and p.vendido == "No"]
    if not activos:
        print(" >> Este usuario no tiene prestamos activos en el sistema.")
        pausa(); return

    print(f"\n Prestamos activos de {usuario.nombre} {usuario.apellido}:")
    print(f"\n {'No.':<5} {'ID Item':<12} {'Nombre Item':<22} {'Fecha Prest.':<14} {'Dias Transc.':<12} {'Plazo':<10} {'Estado Plazo'}")
    print(" " + "-" * 88)

    for idx, p in enumerate(activos, 1):
        item_obj = next((i for i in items if i.item_id == p.item_id), None)
        nombre_it = item_obj.nombre if item_obj else "Desconocido"
        dias_transcurridos = p.dias_prestado()
        plazo = usuario.tiempo
        estado_plazo = "A TIEMPO" if dias_transcurridos <= plazo else "VENCIDO"
        print(f" {idx:<5} {p.item_id:<12} {nombre_it:<22} {p.fecha:<14} {dias_transcurridos:<12} {plazo:<10} {estado_plazo}")

    entrada = input("\n Ingresa el numero o el ID del item a devolver: ").strip().upper()
    prest_sel = next((p for p in activos if p.item_id == entrada), None)

    if prest_sel is None:
        try:
            sel = int(entrada) - 1
            if 0 <= sel < len(activos):
                prest_sel = activos[sel]
        except ValueError:
            pass

    if prest_sel is None:
        print(" >> Seleccion invalida.")
        pausa(); return

    # Registrar Devolución
    fecha_dev = datetime.date.today().strftime("%Y-%m-%d")
    dias_transcurridos = prest_sel.dias_prestado()

    prest_sel.devuelto = "Si"
    prest_sel.fecha_devolucion = fecha_dev

    # Liberar el item en inventario
    item_dev = None
    for i in items:
        if i.item_id == prest_sel.item_id:
            i.disponible = "Si"
            item_dev = i

    guardar_prestamos(prestamos)
    guardar_items(items)

    print("\n OK - Devolucion registrada de manera exitosa.")
    if item_dev:
        print(f"      Item devuelto: {item_dev.nombre} [{item_dev.item_id}]")
    print(f"      Dias que estuvo en prestamo: {dias_transcurridos}")

    # ================================================================
    # GENERACIÓN DEL CERTIFICADO PDF (Usa fuentes compatibles internamente)
    # ================================================================
    print("\n Generando certificado de devolucion...")
    nombre_arch = f"Certificado_Devolucion_{usuario.nombre}_{usuario.apellido}_{prest_sel.item_id}_{fecha_dev}.pdf"
    ruta_pdf = os.path.join(DOC_DIR, nombre_arch)

    f_fuente = 'DejaVu' if _get_font_path() else 'Helvetica'

    pdf = PDFBase()
    pdf.add_page()
    pdf.titulo_seccion('CERTIFICADO DE DEVOLUCION COMPLETADA')
    pdf.ln(2)
    pdf.fila_datos('Fecha Retorno:', fecha_dev)
    pdf.fila_datos('Dias en Posesion:', f"{dias_transcurridos} dias")
    pdf.ln(3)
    pdf.titulo_seccion('DATOS DEL USUARIO REGISTRADO')
    pdf.fila_datos('Nombre Completo:', f"{usuario.nombre} {usuario.apellido}")
    pdf.fila_datos('Documento de Identidad:', usuario.documento)
    pdf.fila_datos('Correo Electronico:', usuario.correo)
    pdf.ln(3)
    pdf.titulo_seccion('DETALLES DEL ITEM DEVUELTO')
    pdf.fila_datos('Nombre del Item:', item_dev.nombre if item_dev else "Desconocido")
    pdf.fila_datos('Identificador ID:', prest_sel.item_id)
    pdf.fila_datos('Categoria:', item_dev.categoria if item_dev else "N/A")
    pdf.fila_datos('Fecha Inicial Prestamo:', prest_sel.fecha)
    pdf.ln(5)
    pdf.set_font(f_fuente, size=10)
    pdf.set_text_color(30, 30, 30)
    pdf.multi_cell(0, 6, 'Por medio de la presente se deja constancia legal de que el item descrito ha sido devuelto fisicamente y reintegrado de forma satisfactoria al inventario disponible de La Reserva.')

    try:
        pdf.output(ruta_pdf)
        _mostrar_link_pdf(ruta_pdf)
    except Exception as e:
        print(f" >> Error al guardar el PDF: {e}")

    # Pausa externa obligatoria para retornar controladamente al menú
    print("")
    pausa()


# ─────────────────────────────────────────────────────────────────
#  MODULO 4: ITEMS CON MAS DE 30 DIAS (FACTURA DE VENTA)
# ─────────────────────────────────────────────────────────────────

def _generar_factura(usuario, items_venta, prestamos_venta, fecha_fact):
    ids = "_".join(p.item_id for p in prestamos_venta)
    nombre_arch = f"{usuario.nombre}_{usuario.apellido}_{ids}.txt"
    ruta = os.path.join(DOC_DIR, nombre_arch)

    subtotal = sum(
        next(i.precio for i in items_venta if i.item_id == p.item_id)
        for p in prestamos_venta
    )
    impuesto = subtotal * 0.23
    total    = subtotal + impuesto

    with open(ruta, "w", encoding="utf-8") as f:
        f.write("=" * 60 + "\n")
        f.write("             FACTURA DE VENTA\n")
        f.write("         LA RESERVA | pf_Algoritmos\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Fecha              : {fecha_fact}\n")
        f.write(f"Cliente            : {usuario.nombre} {usuario.apellido}\n")
        f.write(f"Documento          : {usuario.documento}\n")
        f.write(f"Correo             : {usuario.correo}\n\n")
        f.write("MOTIVO DE VENTA:\n")
        f.write("El item fue prestado por mas de 30 dias sin ser devuelto.\n")
        f.write("Segun el acuerdo con MJ, el prestador debe comprarlo al\n")
        f.write("precio de adquisicion + impuesto por conchudez del 23%.\n\n")
        f.write(f"  {'Item':<24} {'ID':<10} {'Precio':>12}\n")
        f.write("  " + "-" * 48 + "\n")
        for p in prestamos_venta:
            it = next(i for i in items_venta if i.item_id == p.item_id)
            f.write(f"  {it.nombre:<24} {it.item_id:<10}"
                    f" ${it.precio:>11,.0f}\n")
        f.write("  " + "-" * 48 + "\n")
        f.write(f"  {'Subtotal':<35} ${subtotal:>11,.0f}\n")
        f.write(f"  {'Impuesto conchudez (23%)':<35} ${impuesto:>11,.0f}\n")
        f.write(f"  {'TOTAL':<35} ${total:>11,.0f}\n")
        f.write("=" * 60 + "\n")
    return ruta, subtotal, impuesto, total


def consultar_items_30_dias():
    titulo("ITEMS CON MAS DE 30 DIAS - GENERAR VENTA")
    usuarios  = cargar_usuarios()
    items     = cargar_items()
    prestamos = cargar_prestamos()

    vencidos = [p for p in prestamos
                if p.devuelto == "No"
                and p.vendido == "No"
                and p.dias_prestado() > 30]

    if not vencidos:
        print("  No hay items con mas de 30 dias de prestamo.")
        pausa(); return

    print(f"  {len(vencidos)} item(s) con mas de 30 dias:\n")
    print(f"  {'ID':<10} {'Item':<22} {'Usuario':<26} {'Dias':>5}")
    print("  " + "-" * 68)
    for p in vencidos:
        it  = next((i for i in items if i.item_id == p.item_id), None)
        usr = next((u for u in usuarios if u.documento == p.documento), None)
        nombre_it  = it.nombre  if it  else p.item_id
        nombre_usr = f"{usr.nombre} {usr.apellido}" if usr else p.documento
        print(f"  {p.item_id:<10} {nombre_it:<22}"
              f" {nombre_usr:<26} {p.dias_prestado():>5}")

    op = input("\n  Desea generar facturas de venta? (s/n): ").lower()
    if op != "s":
        pausa(); return

    fecha_fact = datetime.date.today().strftime("%Y-%m-%d")
    por_doc = {}
    for p in vencidos:
        por_doc.setdefault(p.documento, []).append(p)

    for doc, lista_p in por_doc.items():
        usr = next((u for u in usuarios if u.documento == doc), None)
        if not usr:
            continue
        its_venta = [i for i in items
                     if any(p.item_id == i.item_id for p in lista_p)]
        ruta, sub, imp, tot = _generar_factura(usr, its_venta, lista_p, fecha_fact)
        for p in lista_p:
            p.vendido = "Si"
            p.devuelto = "Si"
            p.fecha_devolucion = fecha_fact
        for i in items:
            if any(p.item_id == i.item_id for p in lista_p):
                i.disponible = "Si"
        print(f"\n  Factura generada: {ruta}")
        print(f"  Subtotal: ${sub:,.0f} | Impuesto: ${imp:,.0f}"
              f" | TOTAL: ${tot:,.0f}")

    guardar_prestamos(prestamos)
    guardar_items(items)
    pausa()


# ─────────────────────────────────────────────────────────────────
#  MODULO 5: CONSULTAR ARTICULOS PRESTADOS
# ─────────────────────────────────────────────────────────────────

def consultar_articulos_prestados():
    titulo("ARTICULOS PRESTADOS (ordenados mayor a menor por dias)")
    usuarios  = cargar_usuarios()
    items     = cargar_items()
    prestamos = cargar_prestamos()

    activos = [p for p in prestamos
               if p.devuelto == "No" and p.vendido == "No"]
    if not activos:
        print("  No hay articulos prestados actualmente.")
        pausa(); return

    activos.sort(key=lambda p: p.dias_prestado(), reverse=True)

    print(f"  {'Dias':>5}  {'ID':<10} {'Item':<22} {'Usuario':<26} {'Inicio':<12}")
    print("  " + "-" * 80)
    for p in activos:
        it  = next((i for i in items if i.item_id == p.item_id), None)
        usr = next((u for u in usuarios if u.documento == p.documento), None)
        nombre_it  = it.nombre  if it  else p.item_id
        nombre_usr = f"{usr.nombre} {usr.apellido}" if usr else p.documento
        alerta = "  *** ALERTA +20 dias ***" if p.dias_prestado() >= 20 else ""
        print(f"  {p.dias_prestado():>5}  {p.item_id:<10} {nombre_it:<22}"
              f" {nombre_usr:<26} {p.fecha:<12}{alerta}")

    total    = len(activos)
    promedio = sum(p.dias_prestado() for p in activos) / total
    print(f"\n  Total activos: {total}  |  Promedio de dias prestado: {promedio:.1f}")
    pausa()


# ─────────────────────────────────────────────────────────────────
#  MODULO 6: ADMINISTRADOR
#  Acceso: usuario + contrasena
#  Usuario: Disney  |  Contrasena: lareserva2026
# ─────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────
#  REPORTE GENERAL PDF (Administrador)
# ─────────────────────────────────────────────────────────────────

def generar_reporte_pdf():
    # Genera/sobreescribe el reporte PDF consolidado de La Reserva.
    usuarios  = cargar_usuarios()
    items     = cargar_items()
    prestamos = cargar_prestamos()
    fecha_hoy = datetime.date.today().strftime("%Y-%m-%d")

    ruta_pdf = os.path.join(DOC_DIR, "Reporte_LaReserva.pdf")

    total_prest  = len(prestamos)
    activos_n    = sum(1 for p in prestamos if p.devuelto == "No" and p.vendido == "No")
    devueltos    = sum(1 for p in prestamos if p.devuelto == "Si" and p.vendido == "No")
    ventas       = sum(1 for p in prestamos if p.vendido == "Si")
    total_dinero = 0.0
    for p in prestamos:
        if p.vendido == "Si":
            it = next((i for i in items if i.item_id == p.item_id), None)
            if it:
                total_dinero += it.precio * 1.23

    conteo = {u.documento: 0 for u in usuarios}
    for p in prestamos:
        if p.documento in conteo:
            conteo[p.documento] += 1

    fpath = _get_font_path()
    f_fuente = "DejaVu" if (fpath and os.path.exists(fpath)) else "Helvetica"

    pdf = PDFBase()
    pdf.alias_nb_pages()
    pdf.add_page()

    # Titulo principal
    pdf.titulo_seccion("REPORTE GENERAL - LA RESERVA")
    pdf.fila_datos("Fecha de generacion:", fecha_hoy)
    pdf.fila_datos("Generado por:", "Sistema La Reserva | pf_Algoritmos")
    pdf.ln(4)

    # Resumen ejecutivo
    pdf.titulo_seccion("RESUMEN EJECUTIVO")
    pdf.fila_datos("Total prestamos registrados:", str(total_prest))
    pdf.fila_datos("Prestamos activos:",           str(activos_n))
    pdf.fila_datos("Items devueltos:",              str(devueltos))
    pdf.fila_datos("Ventas por +30 dias:",          str(ventas))
    pdf.fila_datos("Total recaudado (ventas):",     "$" + f"{total_dinero:,.0f}")
    pdf.fila_datos("Usuarios registrados:",         str(len(usuarios)))
    pdf.ln(4)

    # Usuarios registrados
    pdf.titulo_seccion("USUARIOS REGISTRADOS")
    if not usuarios:
        pdf.set_font(f_fuente, size=10)
        pdf.cell(0, 7, "Sin usuarios registrados.", 0, 1)
    else:
        enc = "{:<4} {:<18} {:<18} {:<14} {:>5}".format('#','Nombre','Apellido','Documento','Dias')
        pdf.set_font(f_fuente, size=9)
        pdf.set_text_color(80, 80, 80)
        pdf.cell(0, 6, enc, 0, 1)
        pdf.set_text_color(0, 0, 0)
        for idx_u, u in enumerate(usuarios, 1):
            linea = "{:<4} {:<18} {:<18} {:<14} {:>5}".format(idx_u, u.nombre, u.apellido, u.documento, u.tiempo)
            pdf.set_font(f_fuente, size=9)
            pdf.cell(0, 6, linea, 0, 1)
            if idx_u % 30 == 0:
                pdf.add_page()
        if conteo:
            mayor_doc = max(conteo, key=conteo.get)
            menor_doc = min(conteo, key=conteo.get)
            u_may = next((u for u in usuarios if u.documento == mayor_doc), None)
            u_men = next((u for u in usuarios if u.documento == menor_doc), None)
            pdf.ln(3)
            if u_may:
                pdf.fila_datos("Mayor prestamos:", u_may.nombre + " " + u_may.apellido + " (" + str(conteo[mayor_doc]) + ")")
            if u_men:
                pdf.fila_datos("Menor prestamos:", u_men.nombre + " " + u_men.apellido + " (" + str(conteo[menor_doc]) + ")")
    pdf.ln(4)

    # Prestamos activos ordenados por dias
    activos_list = sorted(
        [p for p in prestamos if p.devuelto == "No" and p.vendido == "No"],
        key=lambda p: p.dias_prestado(), reverse=True
    )
    pdf.titulo_seccion("PRESTAMOS ACTIVOS")
    if not activos_list:
        pdf.set_font(f_fuente, size=10)
        pdf.cell(0, 7, "No hay prestamos activos.", 0, 1)
    else:
        enc2 = "{:>5}  {:<10} {:<20} {:<22} {:<12}".format('Dias','ID','Item','Usuario','Inicio')
        pdf.set_font(f_fuente, size=9)
        pdf.set_text_color(80, 80, 80)
        pdf.cell(0, 6, enc2, 0, 1)
        pdf.set_text_color(0, 0, 0)
        for p in activos_list:
            it  = next((i for i in items if i.item_id == p.item_id), None)
            usr = next((u for u in usuarios if u.documento == p.documento), None)
            nom_it  = it.nombre  if it  else p.item_id
            nom_usr = usr.nombre + " " + usr.apellido if usr else p.documento
            alerta = " [+20d]" if p.dias_prestado() >= 20 else ""
            linea2 = "{:>5}  {:<10} {:<20} {:<22} {:<12}{}".format(p.dias_prestado(), p.item_id, nom_it, nom_usr, p.fecha, alerta)
            pdf.set_font(f_fuente, size=9)
            pdf.cell(0, 6, linea2, 0, 1)
    pdf.ln(4)

    # Historial completo de prestamos
    pdf.titulo_seccion("HISTORIAL COMPLETO DE PRESTAMOS")
    enc3 = "{:<10} {:<14} {:<12} {:<5} {:<12} {:<6}".format('ID Item','Documento','Inicio','Dev.','Fecha Dev.','Venta')
    pdf.set_font(f_fuente, size=9)
    pdf.set_text_color(80, 80, 80)
    pdf.cell(0, 6, enc3, 0, 1)
    pdf.set_text_color(0, 0, 0)
    for idx_p, p in enumerate(prestamos, 1):
        fdev = p.fecha_devolucion if p.fecha_devolucion else "-"
        linea3 = "{:<10} {:<14} {:<12} {:<5} {:<12} {:<6}".format(p.item_id, p.documento, p.fecha, p.devuelto, fdev, p.vendido)
        pdf.set_font(f_fuente, size=9)
        pdf.cell(0, 6, linea3, 0, 1)
        if idx_p % 35 == 0:
            pdf.add_page()

    try:
        pdf.output(ruta_pdf)
        print("\n  Reporte PDF generado/actualizado: " + ruta_pdf)
        _mostrar_link_pdf(ruta_pdf)
    except Exception as e:
        print("  >> Error al generar el reporte PDF: " + str(e))
    pausa()

def menu_administrador():
    """
    Acceso al panel de administrador.
    Usuario: Disney
    Contrasena: lareserva2026
    Si los datos no coinciden -> acceso invalido.
    """
    admins = cargar_admins()
    titulo("ACCESO ADMINISTRADOR")
    print("  Ingresa tus credenciales de administrador:")
    print()

    usr_in = input("  Usuario    : ").strip()
    cla_in = input("  Contrasena : ").strip()

    # Validar usuario + contrasena
    if admins.get(usr_in) != cla_in:
        print()
        print("  >> Acceso INVALIDO.")
        print("  >> El usuario o la contrasena son incorrectos.")
        pausa()
        return

    print(f"\n  Bienvenido, {usr_in}!")

    while True:
        banner()
        print(f"  PANEL DE ADMINISTRACION  |  Admin: {usr_in}")
        print(LINEA)
        print("          1. Total de prestamos registrados")
        print("          2. Total de items devueltos")
        print("          3. Total de ventas realizadas")
        print("          4. Total pago recaudado (ventas)")
        print("          5. Lista de usuarios")
        print("          6. Usuario con mayor y menor prestamos")
        print("          7. Generar reporte PDF")
        print("          8. Volver al menu principal")
        print(LINEA)
        op = input("  Opcion: ").strip()

        prestamos = cargar_prestamos()
        usuarios  = cargar_usuarios()
        items     = cargar_items()

        if op == "1":
            titulo("TOTAL DE PRESTAMOS REGISTRADOS")
            total   = len(prestamos)
            activos = sum(1 for p in prestamos if p.devuelto == "No")
            hist    = total - activos
            print(f"  Total historico  : {total}")
            print(f"  Activos          : {activos}")
            print(f"  Finalizados      : {hist}")
            pausa()

        elif op == "2":
            titulo("TOTAL DE ITEMS DEVUELTOS")
            dev = sum(1 for p in prestamos
                      if p.devuelto == "Si" and p.vendido == "No")
            print(f"  Items devueltos voluntariamente: {dev}")
            pausa()

        elif op == "3":
            titulo("TOTAL DE VENTAS REALIZADAS")
            ventas = sum(1 for p in prestamos if p.vendido == "Si")
            print(f"  Total ventas por +30 dias: {ventas}")
            pausa()

        elif op == "4":
            titulo("TOTAL PAGO RECAUDADO (ventas)")
            total_pago = 0.0
            for p in prestamos:
                if p.vendido == "Si":
                    it = next((i for i in items if i.item_id == p.item_id), None)
                    if it:
                        total_pago += it.precio * 1.23
            print(f"  Total recaudado por ventas: ${total_pago:,.0f}")
            pausa()

        elif op == "5":
            titulo("LISTA DE USUARIOS REGISTRADOS")
            if not usuarios:
                print("  No hay usuarios registrados.")
            else:
                print(f"  {'#':<4} {'Nombre':<20} {'Apellido':<20}"
                      f" {'Documento':<16} {'Correo':<30} {'Dias':>5}")
                print("  " + "-" * 96)
                for idx, u in enumerate(usuarios, 1):
                    print(f"  {idx:<4} {u.nombre:<20} {u.apellido:<20}"
                          f" {u.documento:<16} {u.correo:<30} {u.tiempo:>5}")
            pausa()

        elif op == "6":
            titulo("USUARIO CON MAYOR Y MENOR PRESTAMOS")
            if not usuarios:
                print("  No hay usuarios registrados.")
                pausa(); continue
            conteo = {u.documento: 0 for u in usuarios}
            for p in prestamos:
                if p.documento in conteo:
                    conteo[p.documento] += 1
            mayor_doc = max(conteo, key=conteo.get)
            menor_doc = min(conteo, key=conteo.get)
            mayor = next(u for u in usuarios if u.documento == mayor_doc)
            menor = next(u for u in usuarios if u.documento == menor_doc)
            print(f"  Mayor prestamos: {mayor.nombre} {mayor.apellido}"
                  f" ({conteo[mayor_doc]} prestamos)")
            print(f"  Menor prestamos: {menor.nombre} {menor.apellido}"
                  f" ({conteo[menor_doc]} prestamos)")
            pausa()

        elif op == "7":
            titulo("GENERAR REPORTE PDF")
            generar_reporte_pdf()

        elif op == "8":
            break
        else:
            print("  >> Opcion invalida.")
            pausa()


# ─────────────────────────────────────────────────────────────────
#  MENU PRINCIPAL
# ─────────────────────────────────────────────────────────────────

def menu_principal():
    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(DOC_DIR, exist_ok=True)
    inicializar_items()
    while True:
        banner()
        print("          1. Registrar Usuario")
        print("          2. Registrar Prestamo")
        print("          3. Registrar Devolucion")
        print("          4. Consultar Items con mas de 30 dias")
        print("          5. Consultar Articulos Prestados")
        print("          6. Administrador")
        print("          7. Salir")
        print(LINEA)
        op = input("  Selecciona una opcion: ").strip()

        if   op == "1": registrar_usuario()
        elif op == "2": registrar_prestamo()
        elif op == "3": registrar_devolucion()
        elif op == "4": consultar_items_30_dias()
        elif op == "5": consultar_articulos_prestados()
        elif op == "6": menu_administrador()
        elif op == "7":
            print("\n  Hasta pronto! - La Reserva\n")
            break
        else:
            print("  >> Opcion invalida.")
            pausa()





W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
# ─────────────────────────────────────────────────────────────────
#  PUNTO DE ENTRADA
# ─────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    menu_principal()



  ██╗      █████╗     ██████╗ ███████╗███████╗███████╗██████╗ ██╗   ██╗ █████╗
  ██║     ██╔══██╗    ██╔══██╗██╔════╝██╔════╝██╔════╝██╔══██╗██║   ██║██╔══██╗
  ██║     ███████║    ██████╔╝█████╗  ███████╗█████╗  ██████╔╝██║   ██║███████║
  ██║     ██╔══██║    ██╔══██╗██╔══╝  ╚════██║██╔══╝  ██╔══██╗╚██╗ ██╔╝██╔══██║
  ███████╗██║  ██║    ██║  ██║███████╗███████║███████╗██║  ██║  ╚████╔╝██║  ██║
  ╚══════╝╚═╝  ╚═╝    ╚═╝  ╚═╝╚══════╝╚══════╝╚══════╝╚═╝  ╚═╝   ╚═══╝ ╚═╝  ╚═╝


  Bienvenido a La Reserva
          1. Registrar Usuario
          2. Registrar Prestamo
          3. Registrar Devolucion
          4. Consultar Items con mas de 30 dias
          5. Consultar Articulos Prestados
          6. Administrador
          7. Salir


#Manual de Usuario — La Reserva
##Universidad de Antioquia
###Algoritmia y Programación
####Desarrolladores:
####Wbuanderley Ramírez Martínez · Disney Restrepo Enciso

---

###Tabla de Contenido

1. ¿Qué es La Reserva?
2. ¿Qué necesitas para usarlo?
3. ¿Cómo abrir el programa?
4. El menú principal
5. Registrar un usuario
6. Registrar un préstamo
7. Registrar una devolución
8. Consultar ítems con más de 30 días
9. Consultar artículos prestados
10. Panel de Administrador
11. Archivos que genera el sistema
12. Recomendaciones de uso
13. Créditos

---


#1. ¿Qué es La Reserva?
La Reserva es un sistema de gestión de préstamos entre amigos desarrollado en Python. Te permite registrar personas, prestar artículos, controlar devoluciones, generar facturas automáticas y consultar reportes, todo desde una consola fácil de usar.
Si alguna vez olvidaste a quién le prestaste algo, esto es para ti.

#2. ¿Qué necesitas para usarlo?
Antes de empezar, asegúrate de tener lo siguiente:

Python 3.10 o superior instalado en tu computador, o una cuenta activa en Google Colab (gratis, solo necesitas un correo Gmail).
La librería fpdf instalada. El mismo programa la instala automáticamente cuando lo ejecutas en Colab con este comando que ya viene incluido en el notebook:

  pip install fpdf

El archivo .ipynb descargado o en tu Google Drive.


####*Si vas a ejecutarlo en Colab, no tienes que instalar absolutamente nada en tu computador.


#3. ¿Cómo abrir el programa?
###Opción Recomendada — Google Colab

Ve a https://colab.research.google.com
Haz clic en "Subir" y sube el archivo LARESERVADEFINITIVOP.ipynb.
Una vez abierto, haz clic en "Entorno de ejecución" → "Ejecutar todo" (o usa Ctrl + F9).
El programa instalará las dependencias y arrancará solo. Verás el banner de La Reserva en la pantalla.

###Opción alternativa — Local (en tu computador)

Abre una terminal o símbolo del sistema.
Instala la dependencia si no la tienes:

bash   pip install fpdf

Ejecuta el notebook con Jupyter:

bash   jupyter notebook (archivo del código).ipynb

Ejecuta todas las celdas en orden desde la primera.


#4. El menú principal
Cuando el programa arranca, verás el siguiente menú:

          1. Registrar Usuario
          2. Registrar Prestamo
          3. Registrar Devolucion
          4. Consultar Items con mas de 30 dias
          5. Consultar Articulos Prestados
          6. Administrador
          7. Salir
Para elegir una opción, escribe el número correspondiente y presiona Enter.

Si escribes un número que no está en el menú, el sistema te avisará que la opción no es válida y te pedirá que intentes de nuevo.


#5. Registrar un usuario
Elige la opción 1 en el menú principal. El sistema te pedirá que ingreses los siguientes datos: nombre, apellido, número de documento, correo electronico y días a prestar.
Si ingresas un dato incorrecto, el programa te lo dirá y te pedirá que lo corrijas antes de continuar. No pierdes lo que ya escribiste.

####Ejemplo de registro exitoso:
  Nombre    : Laura

  Apellido  : Martinez

  Documento : 1023456789

  Correo    : laura@gmail.com

  Dias de prestamo (5 / 10 / 15 / 30): 15


  OK - Usuario 'Laura Martinez' registrado correctamente.

#6. Registrar un préstamo
Elige la opción 2 en el menú principal.

Pasos:

Escribe el número de documento de la persona que va a recibir el préstamo. Si no está registrada, el sistema te avisará que primero debes registrarla (opción 1).

El sistema te mostrará una lista de todos los ítems disponibles con su ID, nombre, categoría, estado y precio.

Escribe el ID del ítem que deseas prestar (por ejemplo: VID001, LIB002).

El sistema confirmará el préstamo y generará automáticamente un certificado PDF con todos los detalles.


*El certificado se guarda automáticamente en la carpeta /doc con un nombre como:
Certificado_Prestamo_Laura_Martinez_VID001_2026-06-05.pdf

*Un ítem ya prestado no aparece en la lista hasta que sea devuelto.


#7. Registrar una devolución
Elige la opción 3 en el menú principal.
Pasos:

Escribe el documento de la persona que va a devolver el ítem.
El sistema te mostrará una tabla con todos sus préstamos activos, incluyendo cuántos días lleva prestado cada ítem y si ya está vencido o a tiempo. Elige el ítem a devolver escribiendo su ID o su número en la lista.

El sistema registra la devolución y genera un certificado PDF de devolución.


*El certificado se guarda en /doc con un nombre como:
Certificado_Devolucion_Laura_Martinez_VID001_2026-06-05.pdf

*Si la persona no tiene préstamos activos, el sistema te avisará y no te dejará continuar.

#8. Consultar ítems con más de 30 días
Elige la opción 4 en el menú principal.

El sistema busca automáticamente todos los ítems que llevan más de 30 días prestados sin ser devueltos y los muestra en pantalla.

Si hay ítems vencidos, el sistema te preguntará:
  Desea generar facturas de venta? (s/n):
Si escribes s y presionas Enter, el sistema genera una factura de venta en formato .txt por cada usuario que tenga ítems vencidos.

La factura incluye:

Nombre y documento del cliente
Detalle de cada ítem no devuelto
Subtotal del valor de los ítems
Impuesto del 23% (impuesto por conchudez)
Total a pagar

*La factura se guarda en /doc con el nombre:
Laura_Martinez_VID001.txt


#9. Consultar artículos prestados
Elige la opción 5 en el menú principal.
Verás una tabla con todos los préstamos activos, ordenados de mayor a menor cantidad de días prestados. Cada fila muestra:

1. Días transcurridos
2. ID y nombre del ítem
3. Nombre del usuario
4. Fecha de inicio del préstamo
5. Una alerta *** ALERTA +20 dias *** si el ítem lleva 20 días o más

Al final de la lista verás un resumen con el total de préstamos activos y el promedio de días que llevan prestados.

#10. Panel de Administrador
Elige la opción 6 en el menú principal.

El sistema te pedirá usuario y contraseña. Las credenciales válidas son:
Usuarios: Disney, Wbuanderley, John.
Contraseña: lareserva2026

Si escribes mal el usuario o la contraseña, el sistema te denegará el acceso.

Una vez dentro, verás el panel de administración con estas opciones:

          1. Total de prestamos registrados
          2. Total de items devueltos
          3. Total de ventas realizadas
          4. Total pago recaudado (ventas)
          5. Lista de usuarios
          6. Usuario con mayor y menor prestamos
          7. Generar reporte PDF
          8. Volver al menu principal
¿Qué hace cada opción?

1. Muestra cuántos préstamos hay en total (histórico), cuántos están activos y cuántos ya terminaron.
2. Muestra cuántos ítems fueron devueltos voluntariamente (sin necesidad de venta).
3. Muestra cuántos ítems fueron "vendidos" por superar los 30 días.
4. Calcula el dinero total recaudado por todas las ventas (precio del ítem + 23% de impuesto).
5. Lista completa de todos los usuarios registrados con su información.
6. Dice qué usuario tiene más préstamos registrados y cuál tiene menos.
7. Genera (o actualiza) el reporte PDF completo del sistema. Puedes usarlo cuantas veces quieras; cada vez que lo abres refleja el estado actual de La Reserva.
8. Regresa al menú principal.


#11. Archivos que genera el sistema
El programa crea y administra automáticamente estos archivos.

Los archivos se encuentran en las carpetas del Colab.
En Colab, las carpetas /data/ y /doc/ se crean dentro de /content/ automáticamente.


#12. Recomendaciones de uso

El inventario de 20 ítems ya viene cargado la primera vez que ejecutas el programa.

Si ejecutas el programa en Colab, descarga los archivos de /doc/ antes de cerrar la sesión, porque Colab borra los archivos temporales cuando la sesión termina. Esto incluye el reporte PDF del administrador.

Genera el reporte PDF cada vez que hagas cambios importantes (nuevos préstamos, devoluciones, ventas) para tener siempre una copia actualizada descargada.

No modifiques manualmente los archivos .csv de la carpeta /data/, ya que podrías dañar la información del sistema.

Si el programa muestra un error al generar el PDF, revisa que la librería fpdf esté instalada correctamente.


#13. Créditos
La Reserva

Proyecto académico — Algoritmia y Programación 2026-1

Universidad de Antioquia · Facultad de Ingeniería · Ingeniería Industrial

Equipo de desarrollo:

Wbuanderley Ramírez Martínez
Disney Restrepo Enciso

Docente: John Heider Dávila